In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

import torch
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.cuda.set_device(3)
print(f"Using GPU: {torch.cuda.current_device()} "
      f"({torch.cuda.get_device_name(3)})")

Using GPU: 3 (Quadro RTX 6000)


In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib notebook
from argparse import ArgumentParser
import yaml
import os
import math
import torch
# from torch import vmap
from torch.func import vmap, grad
from models import FNN2d
from train_utils import Adam

from solver.BlackScholesEq import BlackScholesEq1D
import traceback

import scipy.io
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import numpy as np
import imageio


from tqdm import tqdm
from train_utils.utils import save_checkpoint, load_checkpoint, update_config, load_config
from train_utils.losses import LpLoss
import train_utils.datasets as datasets
from importlib import reload
reload(datasets)
from train_utils.datasets import DataLoaderBS

try:
    import wandb
except ImportError:
    wandb = None


# Solver Sanity Check

Verify the reference Crank–Nicolson solver before generating training data. We march backward in **τ = T − t** from the terminal payoff (τ = 0) to today's price (τ = T).

In [ ]:
# %matplotlib inline
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Problem parameters
# K = 100.0          # strike
# r = 0.05           # risk-free rate
# sigma = 0.2        # volatility
# T = 1.0            # time to maturity
# S_min, S_max = 1.0, 200.0
# Nx = 256
# dtau = 1e-3

# bs = BlackScholesEq1D(
#     S_min=S_min, S_max=S_max, Nx=Nx,
#     r=r, sigma=sigma, dtau=dtau, T=T,
#     device=device,
# )

# # Terminal payoff is the initial condition in tau coordinates
# v0 = bs.call_payoff(K)
# bc_lo, bc_hi = bs.call_bcs(K)

# # Save ~50 snapshots over [0, T]
# save_interval = max(1, int(T / dtau / 50))
# V = bs.bs_driver(v0, bc_lo, bc_hi, save_interval=save_interval)

# S = bs.S_grid.detach().cpu()
# tau_list = bs.T_list

# print(f'device: {device}')
# print(f'solution shape: {tuple(V.shape)}   # (n_snapshots, Nx)')
# print(f'first tau: {tau_list[0]:.4f}, last tau: {tau_list[-1]:.4f}')

# # Plot a few tau slices
# fig, ax = plt.subplots(figsize=(8, 4))
# for idx in [0, len(V) // 2, -1]:
#     ax.plot(S, V[idx].detach().cpu(), label=fr'$\tau={tau_list[idx]:.3f}$')
# ax.set_xlabel('S')
# ax.set_ylabel('V(S, τ)')
# ax.set_title(f'European call, K={K}, r={r}, σ={sigma}')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()

# # Optional: compare final slice to closed-form Black–Scholes at t=0
# def bs_call_price(S, K, r, sigma, T_rem):
#     """Analytical European call; T_rem = time to maturity."""
#     S = torch.as_tensor(S, dtype=torch.float64)
#     if T_rem <= 0:
#         return torch.clamp(S - K, min=0.0)
#     d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T_rem) / (sigma * math.sqrt(T_rem))
#     d2 = d1 - sigma * math.sqrt(T_rem)
#     from torch.special import ndtr
#     return S * ndtr(d1) - K * math.exp(-r * T_rem) * ndtr(d2)

# V_final = V[-1].detach().cpu()
# V_exact = bs_call_price(S, K, r, sigma, T).cpu()
# rel_err = torch.norm(V_final - V_exact) / torch.norm(V_exact)
# print(f'relative L2 error vs analytical BS at τ=T: {rel_err.item():.4e}')

# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(S, V_final, label='CN solver')
# ax.plot(S, V_exact, '--', label='analytical BS')
# ax.set_xlabel('S')
# ax.set_ylabel('V(S, T)')
# ax.set_title('Today\'s price: solver vs analytical')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()

# Define Loss Functions

## Finite Difference Residual

In [ ]:
def FDM_BlackScholes(u, D=1.0, r=0.05, sigma=0.2, T=1.0, S_min=1.0, S_max=200.0):
    """
    Black-Scholes PDE residual on a normalized (tau_norm, x_norm) grid.

    PDE in log-price x=log(S) and tau=T-t:
        V_tau = a*V_xx + b*V_x - r*V,  a=0.5*sigma^2, b=r-0.5*sigma^2

    DataLoader coordinates: tau_norm in [0,1] -> [0,T], x_norm in [0,1] -> log(S) in [log(S_min), log(S_max)].
    Coefficients are scaled accordingly.

    u     : [batch, Ntau, Nx]
    sigma : scalar OR per-sample tensor of shape [batch] (fix #3).
    Returns residual of shape [batch, Ntau-2, Nx-2].
    """
    batchsize = u.size(0)
    nt = u.size(1)
    nx = u.size(2)
    u = u.reshape(batchsize, nt, nx)

    dt = D / (nt - 1)   # normalized tau spacing
    dx = D / nx          # normalized log-S spacing

    Lx    = math.log(S_max / S_min)

    # Allow sigma to be a per-sample tensor so each option/strike can carry its
    # own volatility.  Reshape to [batch, 1, 1] for broadcasting against the
    # residual interior of shape [batch, Ntau-2, Nx-2].
    if torch.is_tensor(sigma) and sigma.dim() > 0:
        sigma_b = sigma.reshape(batchsize, 1, 1).to(u.device, u.dtype)
    else:
        sigma_b = sigma

    a     = 0.5 * sigma_b ** 2
    b     = r - 0.5 * sigma_b ** 2
    a_eff = a * T / Lx ** 2
    b_eff = b * T / Lx
    r_eff = r * T

    ux  = (u[:, :, 2:] - u[:, :, :-2])                        / (2.0 * dx)
    uxx = (u[:, :, 2:] - 2.0 * u[:, :, 1:-1] + u[:, :, :-2]) / dx ** 2
    utau = (u[:, 2:, :] - u[:, :-2, :]) / (2.0 * dt)

    Du = (utau[:, :, 1:-1]
          - a_eff * uxx[:, 1:-1, :]
          - b_eff * ux[:, 1:-1, :]
          + r_eff * u[:, 1:-1, 1:-1])
    return Du


def PINO_loss_bs(u, u0, r=0.05, sigma=0.2, T=1.0, S_min=1.0, S_max=200.0):
    """
    PINO loss: IC matching + Black-Scholes PDE residual.

    u     : [batch, Ntau, Nx]  full solution predicted by model
    u0    : [batch, Nx]        payoff (initial condition at tau=0)
    sigma : scalar OR per-sample tensor [batch] (fix #3).
    """
    batchsize = u.size(0)
    nt = u.size(1)
    nx = u.size(2)
    u = u.reshape(batchsize, nt, nx)

    loss_u = F.mse_loss(u[:, 0, :], u0)

    Du = FDM_BlackScholes(u, r=r, sigma=sigma, T=T, S_min=S_min, S_max=S_max)
    loss_f = F.mse_loss(Du, torch.zeros_like(Du))

    return loss_u, loss_f


def sigma_from_channel(x, sigma_min, sigma_max):
    """
    Recover the per-sample physical volatility from the (normalized) sigma
    input channel.  x is the model input [batch, Ntau, Nx, C]; the sigma
    channel is the last one (index 3) and is constant across the grid.
    Returns a tensor of shape [batch], or None if there is no sigma channel.
    """
    if x.shape[-1] < 4:
        return None
    sig_norm = x[:, 0, 0, 3]
    return sigma_min + sig_norm * (sigma_max - sigma_min)

# Define Training Function

In [ ]:
def train_bs(model,
             train_loader,
             optimizer,
             scheduler,
             config,
             rank=0,
             log=False,
             project='PINO-BlackScholes-default',
             group='default',
             tags=['default'],
             use_tqdm=True):
    if rank == 0 and wandb and log:
        run = wandb.init(project=project,
                         entity='shawngr2',
                         group=group,
                         config=config,
                         tags=tags, reinit=True,
                         settings=wandb.Settings(start_method='fork'))

    data_weight = config['train']['xy_loss']
    f_weight    = config['train']['f_loss']
    ic_weight   = config['train']['ic_loss']
    r     = config['data']['r']
    sigma = config['data']['sigma']
    T     = config['data']['T']
    S_min = config['data']['S_min']
    S_max = config['data']['S_max']
    sigma_min = config['data'].get('sigma_min', sigma)
    sigma_max = config['data'].get('sigma_max', sigma)
    ckpt_freq = config['train']['ckpt_freq']

    model.train()
    myloss = LpLoss(size_average=True)
    pbar = range(config['train']['epochs'])
    if use_tqdm:
        pbar = tqdm(pbar, dynamic_ncols=True, smoothing=0.1)

    for e in pbar:
        model.train()
        train_pino = 0.0
        data_l2    = 0.0
        train_ic   = 0.0
        train_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(rank), y.to(rank)
            out = model(x).reshape(y.shape)
            data_loss = myloss(out, y)

            # per-sample sigma from the input channel (falls back to scalar)
            sigma_b = sigma_from_channel(x, sigma_min, sigma_max)
            if sigma_b is None:
                sigma_b = sigma
            loss_ic, loss_f = PINO_loss_bs(
                out, x[:, 0, :, 0],
                r=r, sigma=sigma_b, T=T, S_min=S_min, S_max=S_max)
            total_loss = loss_ic * ic_weight + loss_f * f_weight + data_loss * data_weight

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            data_l2    += data_loss.item()
            train_pino += loss_f.item()
            train_loss += total_loss.item()
            train_ic   += loss_ic.item()

        scheduler.step()
        data_l2    /= len(train_loader)
        train_pino /= len(train_loader)
        train_loss /= len(train_loader)
        train_ic   /= len(train_loader)

        if use_tqdm:
            pbar.set_description(
                f'Epoch {e}, train loss: {train_loss:.5f} '
                f'train f error: {train_pino:.5f}; '
                f'data l2 error: {data_l2:.5f}; '
                f'train ic error: {train_ic:.5f}'
            )
        if wandb and log:
            wandb.log({
                'Train f error':  train_pino,
                'Train L2 error': data_l2,
                'Train ic error': train_ic,
                'Train loss':     train_loss,
            })

        if e % ckpt_freq == 0:
            save_checkpoint(config['train']['save_dir'],
                            config['train']['save_name'].replace('.pt', f'_{e}.pt'),
                            model, optimizer)

    save_checkpoint(config['train']['save_dir'],
                    config['train']['save_name'],
                    model, optimizer)
    print('Done!')

# Evaluation Function

In [ ]:
def eval_bs(model,
            dataloader,
            config,
            device,
            use_tqdm=True):
    model.eval()
    myloss = LpLoss(size_average=True)
    r     = config['data']['r']
    sigma = config['data']['sigma']
    T     = config['data']['T']
    S_min = config['data']['S_min']
    S_max = config['data']['S_max']
    sigma_min = config['data'].get('sigma_min', sigma)
    sigma_max = config['data'].get('sigma_max', sigma)

    pbar = tqdm(dataloader, dynamic_ncols=True, smoothing=0.05) if use_tqdm else dataloader

    test_err = []
    f_err    = []

    with torch.no_grad():
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            out = model(x).reshape(y.shape)
            data_loss = myloss(out, y)

            sigma_b = sigma_from_channel(x, sigma_min, sigma_max)
            if sigma_b is None:
                sigma_b = sigma
            _, f_loss = PINO_loss_bs(
                out, x[:, 0, :, 0],
                r=r, sigma=sigma_b, T=T, S_min=S_min, S_max=S_max)
            test_err.append(data_loss.item())
            f_err.append(f_loss.item())

    mean_f_err = np.mean(f_err)
    std_f_err  = np.std(f_err, ddof=1) / np.sqrt(len(f_err))
    mean_err   = np.mean(test_err)
    std_err    = np.std(test_err, ddof=1) / np.sqrt(len(test_err))

    print(f'==Averaged relative L2 error mean: {mean_err}, std error: {std_err}==\n'
          f'==Averaged equation error mean: {mean_f_err}, std error: {std_f_err}==')

# Save and Load Data

In [ ]:
def save_data(data_path, test_x, test_y, preds_y):
    data_dir, _ = os.path.split(data_path)
    os.makedirs(data_dir, exist_ok=True)
    np.savez(data_path, test_x=test_x, test_y=test_y, preds_y=preds_y)

def load_data(data_path):
    data = np.load(data_path)
    return data['test_x'], data['test_y'], data['preds_y']

# Visualization

In [ ]:
def plot_predictions(key, test_x, test_y, preds_y,
                     S_min=1.0, S_max=200.0,
                     save_path=None, font_size=None):
    if font_size is not None:
        plt.rcParams.update({'font.size': font_size})

    pred = preds_y[key]
    true = test_y[key]
    a    = test_x[key]

    Ntau, Nx, _ = a.shape
    payoff  = a[0, :, 0]
    x_norm  = a[0, :, 1]
    tau_norm = a[:, 0, 2]

    Lx = math.log(S_max / S_min)
    S  = np.exp(np.log(S_min) + x_norm * Lx)

    slice_indices = sorted({0, Ntau // 2, Ntau - 1})
    slice_labels = [f'$tau={tau_norm[i]:.3f}$' for i in slice_indices]

    fig = plt.figure(figsize=(23, 5))

    plt.subplot(1, 4, 1)
    plt.plot(S, payoff)
    plt.xlabel('$S$')
    plt.ylabel('$V$')
    plt.title('Payoff $V(S, tau=0)$')
    plt.xlim([S_min, S_max])
    plt.tight_layout()

    def plot_line_panel(values, title, ylabel):
        for idx, label in zip(slice_indices, slice_labels):
            plt.plot(S, values[idx], label=label)
        plt.xlabel('$S$')
        plt.ylabel(ylabel)
        plt.title(title)
        plt.xlim([S_min, S_max])
        plt.grid(alpha=0.25)
        plt.legend(fontsize=8)
        plt.tight_layout()

    plt.subplot(1, 4, 2)
    plot_line_panel(true, 'Exact $V(S, tau)$', '$V$')

    plt.subplot(1, 4, 3)
    plot_line_panel(pred, 'Predict $V(S, tau)$', '$V$')

    plt.subplot(1, 4, 4)
    plot_line_panel(pred - true, 'Absolute Error', 'Error')

    if save_path is not None:
        if save_path.endswith((os.sep, '/', '\\')) or os.path.isdir(save_path):
            os.makedirs(save_path, exist_ok=True)
            save_file = os.path.join(save_path, f'plot_predictions_{key}.png')
        else:
            os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
            save_file = save_path if save_path.lower().endswith('.png') else f'{save_path}.png'
        plt.savefig(save_file, bbox_inches='tight')
    # plt.show()

# Movies

In [ ]:
def generate_movie_1D(key, test_x, test_y, preds_y,
                      S_min=1.0, S_max=200.0,
                      plot_title='Black-Scholes',
                      movie_dir='', movie_name='movie.gif',
                      frame_basename='movie', frame_ext='jpg',
                      remove_frames=True, font_size=None):
    frame_files = []
    os.makedirs(movie_dir, exist_ok=True)
    if font_size is not None:
        plt.rcParams.update({'font.size': font_size})

    pred = preds_y[key]
    true = test_y[key]
    a    = test_x[key]

    Ntau, Nx, _ = a.shape
    x_norm   = a[0, :, 1]
    tau_norm = a[:, 0, 2]
    Lx = math.log(S_max / S_min)
    S  = np.exp(np.log(S_min) + x_norm * Lx)

    fig = plt.figure(figsize=(6, 5))
    ax  = fig.add_subplot(111)
    plt.ion()
    fig.show()
    fig.canvas.draw()

    ax.plot(S, true[0], 'b-',  label='Exact')
    ax.plot(S, pred[0], 'r--', label='PINO Prediction')
    ylim = plt.ylim()
    plt.tight_layout()

    for i in range(Ntau):
        ax.clear()
        ax.plot(S, true[i], 'b-',  label='Exact')
        ax.plot(S, pred[i], 'r--', label='PINO Prediction')
        plt.xlabel('$S$')
        plt.ylabel('$V$')
        plt.title(f'{plot_title}  tau={tau_norm[i]:.2f}')
        plt.legend(loc='upper left')
        plt.ylim(ylim)
        plt.xlim([S_min, S_max])
        plt.tight_layout()
        fig.canvas.draw()

        if movie_dir:
            frame_path = os.path.join(movie_dir, f'{frame_basename}-{i:03}.{frame_ext}')
            frame_files.append(frame_path)
            plt.savefig(frame_path)

    if movie_dir:
        movie_path = os.path.join(movie_dir, movie_name)
        with imageio.get_writer(movie_path, mode='I') as writer:
            for frame in frame_files:
                writer.append_data(imageio.imread(frame))

    if movie_dir and remove_frames:
        for frame in frame_files:
            try:
                os.remove(frame)
            except Exception:
                pass

# Load Config File

In [ ]:
config_file = 'configs/custom/black_scholes-0000.yaml'
config = load_config(config_file)
display(config)

# Parameters

In [ ]:
Nsamples = config['data']['total_num']
N        = config['data']['nx']
Nt0      = config['data']['nt']
sub_x    = config['data']['sub']
sub_t    = config['data']['sub_t']
Nx       = N // sub_x
Nt       = Nt0 // sub_t + 1

r     = config['data']['r']
sigma = config['data']['sigma']
T     = config['data']['T']
S_min = config['data']['S_min']
S_max = config['data']['S_max']
K_min = config['data']['K_min']
K_max = config['data']['K_max']

# Volatility is now an input dimension (fix #3): the synthetic set spans a
# grid of sigma slices instead of a single fixed sigma.
sigma_min = config['data']['sigma_min']
sigma_max = config['data']['sigma_max']
n_sigma   = config['data']['n_sigma']
n_K       = Nsamples // n_sigma
assert n_sigma * n_K == Nsamples, \
    f"total_num ({Nsamples}) must equal n_sigma ({n_sigma}) * n_K ({n_K})"

# Solver step size and snapshot interval.
# Pick steps so the saved snapshots line up EXACTLY with the Nt grid the
# DataLoader expects (steps_total = Nt0 * steps_per_snap -> Nt0+1 snapshots
# spanning [0, T]).  This avoids tau-grid aliasing for any T.
steps_per_snap = 5
steps_total    = Nt0 * steps_per_snap
dtau_solve     = T / steps_total
save_int       = steps_per_snap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'Nx={Nx}, Nt={Nt}, save_int={save_int}, dtau_solve={dtau_solve:.2e}')
print(f'sigma grid: {n_sigma} values in [{sigma_min}, {sigma_max}], '
      f'{n_K} strikes each -> {Nsamples} samples')

# Generate Dataset

Sample `Nsamples` strike prices K uniformly from [K_min, K_max],
compute the European call payoff for each K, and run the Crank-Nicolson solver.

In [ ]:
# Sample grids: the dataset is a Cartesian product of sigma slices x strikes.
# Each (sigma, K) pair becomes one training sample; sigma rides along as an
# extra input channel so the operator can price ANY volatility (fix #3).
sigma_samples  = torch.linspace(sigma_min, sigma_max, n_sigma)
K_samples_base = torch.linspace(K_min, K_max, n_K)
print(f"sigma slices: {[round(float(s), 3) for s in sigma_samples]}")
print(f"strikes per slice: {n_K} in [{K_min}, {K_max}]")

In [ ]:
import torch
import math

# ------------------------------------------------------------------
# Batched Crank-Nicolson generator.  Runs ALL strikes for a single
# sigma at once; we loop over sigma slices on top (fix #3).
# ------------------------------------------------------------------
def batched_thomas_solve(a_sub, a_diag, a_super, d):
    """Solve a tridiagonal system Ax = d simultaneously across a batch."""
    n = d.shape[1]
    batch_size = d.shape[0]
    c_p = torch.empty(n - 1, device=d.device, dtype=d.dtype)
    d_p = torch.empty(batch_size, n, device=d.device, dtype=d.dtype)
    c_p[0]    = a_super[0] / a_diag[0]
    d_p[:, 0] = d[:, 0] / a_diag[0]
    for i in range(1, n):
        denom = a_diag[i] - a_sub[i - 1] * c_p[i - 1]
        if i < n - 1:
            c_p[i] = a_super[i] / denom
        d_p[:, i] = (d[:, i] - a_sub[i - 1] * d_p[:, i - 1]) / denom
    x = torch.empty(batch_size, n, device=d.device, dtype=d.dtype)
    x[:, -1] = d_p[:, -1]
    for i in range(n - 2, -1, -1):
        x[:, i] = d_p[:, i] - c_p[i] * x[:, i + 1]
    return x


def generate_bs_batch(solver, K_samples, save_interval):
    """
    Solve the BS PDE for every strike in K_samples at the solver's sigma.
    Returns
        a_gpu : [n_K, Nx]            payoffs (initial conditions)
        u_gpu : [n_K, Nsnap, Nx]     price surfaces (Nsnap = Nt0 + 1)
    """
    S_grid = solver.S_grid
    dtau   = solver.dtau
    tend   = solver.tend
    r_     = solver.r
    S_hi   = S_grid[-1]
    K_b    = K_samples.to(device=solver.device, dtype=solver.dtype)

    a_gpu = torch.clamp(S_grid.unsqueeze(0) - K_b.unsqueeze(1), min=0.0)  # [n_K, Nx]

    tau_steps = []
    current_tau = 0.0
    while current_tau < tend - 1e-12:
        tau_steps.append(current_tau)
        current_tau += dtau
    tau_steps = torch.tensor(tau_steps, device=solver.device, dtype=solver.dtype)
    N_time_steps = len(tau_steps)

    V_hi_all = S_hi - K_b.unsqueeze(1) * torch.exp(-r_ * tau_steps.unsqueeze(0))

    v_current = a_gpu.clone()
    v_current[:, 0]  = 0.0
    v_current[:, -1] = S_hi - K_b

    V_history = []
    step_idx = 0
    if save_interval != 0 and step_idx % save_interval == 0:
        V_history.append(v_current.clone())

    alpha    = solver.alpha
    gamma    = solver.gamma
    B_matrix = solver.B

    for step_idx in range(N_time_steps):
        tau_new = tau_steps[step_idx] + dtau
        v_lo_old = v_lo_new = 0.0
        v_hi_old = V_hi_all[:, step_idx]
        if step_idx + 1 < N_time_steps:
            v_hi_new = V_hi_all[:, step_idx + 1]
        else:
            v_hi_new = S_hi - K_b * math.exp(-r_ * (tau_new.item()))

        v_in = v_current[:, 1:-1]
        rhs  = torch.matmul(v_in, B_matrix.t())
        rhs[:, 0]  = rhs[:, 0]  + 0.5 * dtau * alpha * (v_lo_old + v_lo_new)
        rhs[:, -1] = rhs[:, -1] + 0.5 * dtau * gamma * (v_hi_old + v_hi_new)

        v_new_in = batched_thomas_solve(solver.a_sub, solver.a_diag, solver.a_super, rhs)
        v_next = torch.empty_like(v_current)
        v_next[:, 0]    = v_lo_new
        v_next[:, -1]   = v_hi_new
        v_next[:, 1:-1] = v_new_in
        v_current = v_next

        loop_step = step_idx + 1
        if save_interval != 0 and loop_step % save_interval == 0:
            V_history.append(v_current.clone())

    u_gpu = torch.stack(V_history).permute(1, 0, 2)  # [n_K, Nsnap, Nx]
    return a_gpu, u_gpu


# ------------------------------------------------------------------
# Loop over sigma slices and assemble the full dataset.
# ------------------------------------------------------------------
A0_list, U_list, SIG_list = [], [], []
for sig in tqdm(sigma_samples, desc='Generating BS data (sigma slices)'):
    sig_val = float(sig)
    solver = BlackScholesEq1D(
        S_min=S_min, S_max=S_max, Nx=N,
        r=r, sigma=sig_val, dtau=dtau_solve, T=T,
        device=device,
    )
    a_sig, u_sig = generate_bs_batch(solver, K_samples_base, save_int)
    A0_list.append(a_sig.cpu().float())
    U_list.append(u_sig.cpu().float())
    SIG_list.append(torch.full((n_K,), sig_val, dtype=torch.float32))

a = torch.cat(A0_list, dim=0)                  # [Nsamples, Nx]
u = torch.cat(U_list, dim=0)                   # [Nsamples, Nsnap, Nx]
sigma_per_sample = torch.cat(SIG_list, dim=0)  # [Nsamples]

# Shuffle so the train/test split spans every sigma slice.
perm = torch.randperm(a.shape[0])
a = a[perm]
u = u[perm]
sigma_per_sample = sigma_per_sample[perm]

print('Generation Complete!')
display(a.shape, u.shape, sigma_per_sample.shape)


In [ ]:
dataset = DataLoaderBS(a, u, config['data']['nx'], config['data']['nt'],
                       config['data']['sub'], config['data']['sub_t'],
                       sigma_data=sigma_per_sample,
                       sigma_min=config['data']['sigma_min'],
                       sigma_max=config['data']['sigma_max'])
train_loader = dataset.make_loader(config['data']['n_train'],
                                   config['train']['batchsize'],
                                   start=0, train=True)
test_loader  = dataset.make_loader(config['data']['n_test'],
                                   config['test']['batchsize'],
                                   start=config['data']['n_train'], train=False)

In [ ]:
log = False
# in_dim=4: (payoff, x_norm, tau_norm, sigma_norm)  -- sigma is now an input (fix #3)
model = FNN2d(modes1=config['model']['modes1'],
              modes2=config['model']['modes2'],
              fc_dim=config['model']['fc_dim'],
              layers=config['model']['layers'],
              in_dim=4,
              activation=config['model']['activation']).to(device)

optimizer = Adam(model.parameters(), betas=(0.9, 0.999),
                 lr=config['train']['base_lr'])
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=config['train']['milestones'],
    gamma=config['train']['scheduler_gamma'])

# Load from Checkpoint

In [ ]:
load_checkpoint(model, ckpt_path=config['train']['ckpt'], optimizer=None)

# Train the Model

In [ ]:
train_bs(model,
         train_loader,
         optimizer,
         scheduler,
         config,
         rank=device,
         log=log,
         project=config['log']['project'],
         group=config['log']['group'])

# Evaluate on Test Data

In [ ]:
eval_bs(model, test_loader, config, device)

In [ ]:
from real_data_validation import (
    fetch_aapl_options, filter_liquid_calls, split_calls_train_val,
    calibrate_sigma_per_expiry,
    build_sparse_training_set, DataLoaderBSSparse, train_bs_real,
)

# NOTE: the S/K/T domain now lives in the config and is shared with the
# synthetic training set (fix #4), so there is NO post-hoc override here.
# Make sure config S_min/S_max bracket the underlying spot and the real
# strikes BEFORE generating the synthetic data + training above.

df = fetch_aapl_options('AAPL', max_expirations=6)
calls = filter_liquid_calls(df, moneyness_range=(0.90, 1.10))

# fix #1: split into disjoint train / validation BEFORE anything else.
train_calls, val_calls = split_calls_train_val(calls, val_frac=0.30, seed=0)

# fix #2: calibrate ONE Black-Scholes sigma per expiry on the TRAIN set only.
# This same sigma is fed to PINO (via its channel) and to analytical BS.
sigma_lookup = calibrate_sigma_per_expiry(train_calls, r=config['data']['r'])

# Optional: sparse fine-tuning on the TRAIN calls. Sigma enters through the
# input channel, so the PDE residual now matches each option's calibrated vol
# instead of a hard-coded 0.2 (fix #3).
x_data, y_sparse, mask, sigma_data, meta = build_sparse_training_set(
    train_calls, config, sigma_lookup=sigma_lookup)

sparse_dataset = DataLoaderBSSparse(
    x_data, y_sparse, mask,
    config['data']['nx'], config['data']['nt'],
    config['data']['sub'], config['data']['sub_t'],
    sigma_data=sigma_data,
    sigma_min=config['data']['sigma_min'],
    sigma_max=config['data']['sigma_max'])

train_loader = sparse_dataset.make_loader(
    len(meta), config['train']['batchsize'], start=0, train=True)

train_bs_real(model, train_loader, optimizer, scheduler, config, rank=device)

In [ ]:
from real_data_validation import (
    build_validation_set, evaluate_pino_vs_market,
    plot_validation_results, plot_vol_smile_comparison,
)

# Validate on the HELD-OUT calls (fix #1), priced at the train-calibrated
# per-expiry sigma (fix #2/#3).  evaluate_pino_vs_market now also reports the
# PINO-vs-BS gap (operator fidelity) alongside the vs-market errors.
val_x, val_meta = build_validation_set(val_calls, config, sigma_lookup=sigma_lookup)
results = evaluate_pino_vs_market(model, val_x, val_meta, config, device)
plot_validation_results(results, save_dir='BlackScholes/figures/')
plot_vol_smile_comparison(results, save_dir='BlackScholes/figures/')

In [ ]:
Nx = config['data']['nx'] // config['data']['sub']
Nt = config['data']['nt'] // config['data']['sub_t'] + 1
Ntest = config['data']['n_test']
in_dim = getattr(model, 'in_dim', 3)   # 4 now that sigma is a channel
model.eval()
test_x = np.zeros((Ntest,Nt,Nx,in_dim))
preds_y = np.zeros((Ntest,Nt,Nx))
test_y = np.zeros((Ntest,Nt,Nx))
with torch.no_grad():
    for i, data in enumerate(test_loader):
        data_x, data_y = data
        data_x, data_y = data_x.to(device), data_y.to(device)
        pred_y = model(data_x).reshape(data_y.shape)
        test_x[i] = data_x.cpu().numpy()
        test_y[i] = data_y.cpu().numpy()
        preds_y[i] = pred_y.cpu().numpy()
#     data_loss = myloss(out, y)

In [ ]:
plot_predictions(key, test_x, test_y, preds_y, 1, 200, save_path="BlackScholes/figures/")

In [ ]:
generate_movie_1D(key, test_x, test_y, preds_y,
                      S_min=1.0, S_max=200.0,
                      plot_title='Black-Scholes',
                      movie_dir='BlackScholes/movies/', movie_name='movie.gif',
                      frame_basename='movie', frame_ext='jpg',
                      remove_frames=True, font_size=None)